In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
# Dates management
import datetime

# Visualization library
import altair as alt
import matplotlib.pyplot as plt

# Data manipulation library
import pandas as pd

# Enable Altair to display all rows of data
alt.data_transformers.enable('default', max_rows=None)

DataTransformerRegistry.enable('default')

In [19]:
df_person_fix = pd.read_pickle("datap/df_person_fix.pkl")
df_bio=pd.read_pickle("datap/df_bio.pkl")
df_visit=pd.read_pickle("datap/df_visit.pkl")

In [9]:
df_bio.info()
df_bio.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6188 entries, 0 to 6187
Data columns (total 6 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   measurement_id        6188 non-null   float64       
 1   visit_occurrence_id   6188 non-null   float64       
 2   measurement_datetime  6188 non-null   datetime64[ns]
 3   concept_source_value  6188 non-null   object        
 4   transformed_value     6188 non-null   float64       
 5   transformed_unit      6188 non-null   object        
dtypes: datetime64[ns](1), float64(3), object(2)
memory usage: 290.2+ KB


,measurement_id,visit_occurrence_id,measurement_datetime,concept_source_value,transformed_value,transformed_unit
0,85402828.0,87578820.0,2022-05-06,crp,4.53,mg/L
1,89452956.0,82594821.0,2024-10-23,hb,14.18,g/dL
2,88918901.0,87469572.0,2020-02-23,hb,12.31,g/dL
3,86302942.0,88399883.0,2020-02-28,hb,12.04,g/dL
4,86607398.0,86752406.0,2025-08-28,crp,3.86,mg/L


In [13]:

df_bio["concept_source_value"].value_counts()


concept_source_value
crp     1547
hb      1547
bmi     1547
urea    1547
Name: count, dtype: int64

In [14]:
df_bio["transformed_unit"].value_counts()

transformed_unit
mg/L        1547
g/dL        1547
kg.cm^-2    1547
mmol/L      1547
Name: count, dtype: int64

In [16]:
measurement_summary = df_bio.groupby("measurement_datetime", as_index=False).visit_occurrence_id.count()

In [17]:
alt.Chart(measurement_summary).mark_bar(size=1).encode(
    alt.X('measurement_datetime:T', scale=alt.Scale(padding=5)),
    y='visit_occurrence_id:Q'
)

alt.Chart(...)

In [3]:
df_dedup_det = pd.read_pickle('datap/df_dedup_deterministic.pkl')
df_dedup_det.info()
df_dedup_det.head()

<class 'pandas.core.frame.DataFrame'>
Index: 36 entries, 9 to 28
Data columns (total 2 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   unique_person_id  36 non-null     float64
 1   person_id         36 non-null     int64  
dtypes: float64(1), int64(1)
memory usage: 864.0 bytes


,unique_person_id,person_id
9,84763748.0,82185488
36,81299393.0,87690370
29,81076351.0,81292219
0,83290919.0,86711549
0,89552692.0,80057927


In [6]:
# Outer Join
df_person_dedup_det = pd.merge(df_person_fix, df_dedup_det, on = 'person_id', how = 'outer')

# Complete the unique_person_id column
df_person_dedup_det['unique_person_id'] = df_person_dedup_det['unique_person_id'].fillna(df_person_dedup_det['person_id'])

# Only keep one row per patient
df_person_dedup_det = df_person_dedup_det.drop_duplicates(['unique_person_id'], keep = 'first')

In [7]:
print(f"We have {df_person_dedup_det.unique_person_id.nunique()} unique patient ids in this dataset when using the determinist algorithm.")

We have 1494 unique patient ids in this dataset when using the determinist algorithm.


In [20]:
df_bio_fix=pd.merge(df_visit,df_bio,on="visit_occurrence_id",how="inner")

In [21]:
df_bio_fix.info()
df_bio_fix.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6188 entries, 0 to 6187
Data columns (total 11 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   visit_occurrence_id   6188 non-null   float64       
 1   care_site_id          6188 non-null   object        
 2   visit_start_datetime  6188 non-null   datetime64[ns]
 3   visit_end_datetime    6184 non-null   datetime64[ns]
 4   visit_source_value    6188 non-null   object        
 5   person_id             6188 non-null   float64       
 6   measurement_id        6188 non-null   float64       
 7   measurement_datetime  6188 non-null   datetime64[ns]
 8   concept_source_value  6188 non-null   object        
 9   transformed_value     6188 non-null   float64       
 10  transformed_unit      6188 non-null   object        
dtypes: datetime64[ns](3), float64(4), object(4)
memory usage: 531.9+ KB


,visit_occurrence_id,care_site_id,visit_start_datetime,visit_end_datetime,visit_source_value,person_id,measurement_id,measurement_datetime,concept_source_value,transformed_value,transformed_unit
0,88801206.0,Centre F.Sinoussi,2020-01-24,2020-01-26,Hospitalisés,82723807.0,81228660.0,2020-01-24,crp,4.53,mg/L
1,88801206.0,Centre F.Sinoussi,2020-01-24,2020-01-26,Hospitalisés,82723807.0,89065276.0,2020-01-24,urea,3.71,mmol/L
2,88801206.0,Centre F.Sinoussi,2020-01-24,2020-01-26,Hospitalisés,82723807.0,85894139.0,2020-01-24,bmi,27.84,kg.cm^-2
3,88801206.0,Centre F.Sinoussi,2020-01-24,2020-01-26,Hospitalisés,82723807.0,85641747.0,2020-01-24,hb,11.04,g/dL
4,81327360.0,Centre F.Sinoussi,2020-02-25,2020-03-05,Hospitalisés,80260379.0,85453397.0,2020-02-25,hb,15.00,g/dL


In [22]:
df_bio_fix_final = pd.merge(df_person_dedup_det, df_bio_fix, on = 'person_id', how = 'inner')

In [23]:
df_bio_fix_final.info()
df_bio_fix_final.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5976 entries, 0 to 5975
Data columns (total 16 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   birth_datetime        5976 non-null   datetime64[ns]
 1   death_datetime        3276 non-null   datetime64[ns]
 2   gender_source_value   5976 non-null   object        
 3   cdm_source            5976 non-null   object        
 4   person_id             5976 non-null   float64       
 5   unique_person_id      5976 non-null   float64       
 6   visit_occurrence_id   5976 non-null   float64       
 7   care_site_id          5976 non-null   object        
 8   visit_start_datetime  5976 non-null   datetime64[ns]
 9   visit_end_datetime    5972 non-null   datetime64[ns]
 10  visit_source_value    5976 non-null   object        
 11  measurement_id        5976 non-null   float64       
 12  measurement_datetime  5976 non-null   datetime64[ns]
 13  concept_source_val

,birth_datetime,death_datetime,gender_source_value,cdm_source,person_id,unique_person_id,visit_occurrence_id,care_site_id,visit_start_datetime,visit_end_datetime,visit_source_value,measurement_id,measurement_datetime,concept_source_value,transformed_value,transformed_unit
0,1961-07-09,2022-02-20,female,EHR 1,80006137.0,80006137.0,87721793.0,Clinique L.Pasteur,2022-02-19,2022-02-20,Hospitalisés,84366354.0,2022-02-19,crp,4.47,mg/L
1,1961-07-09,2022-02-20,female,EHR 1,80006137.0,80006137.0,87721793.0,Clinique L.Pasteur,2022-02-19,2022-02-20,Hospitalisés,83248298.0,2022-02-19,bmi,21.79,kg.cm^-2
2,1961-07-09,2022-02-20,female,EHR 1,80006137.0,80006137.0,87721793.0,Clinique L.Pasteur,2022-02-19,2022-02-20,Hospitalisés,81724471.0,2022-02-19,urea,3.85,mmol/L
3,1961-07-09,2022-02-20,female,EHR 1,80006137.0,80006137.0,87721793.0,Clinique L.Pasteur,2022-02-19,2022-02-20,Hospitalisés,80756786.0,2022-02-19,hb,13.10,g/dL
4,1951-12-31,2024-07-28,female,EHR 1,80021799.0,80021799.0,80105823.0,Hopital M.Bres,2024-07-24,2024-07-28,Hospitalisés,86628185.0,2024-07-24,bmi,24.92,kg.cm^-2
